# Agent Evaluation

Code hóa trajectory scoring function: Score = task_success + answer_quality + tool_accuracy - unnecessary_steps - latency - cost

Điểm mấu chốt: unnecessary_steps không được gán bằng một con số cố định - nó phải được tính thật từ trajectory (phát hiện tool call trùng lặp/thừa), và latency/cost được cộng dồn thật từ chi phí từng bước, chứ không random hay gõ tay theo kết luận có sẵn.

## 1. Biểu diện trajectory + các hàm tính thật
- detect_unnecessary_steps: một bước bị coi là thừa nếu nó gọi cùng tool với số query/args một bước nào đó đã gọi trước đó trong trajectory (redundant call thật sự, không đóng góp thông tin mới).
- compute_latency / compute_cost: cộng dồn từ bảng chi phí trung bình thực tế của từng loại tool - nhiều bước hơn thì latency/cost thật sự cao hơn, không phải con số bịa.



In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Step:
    tool: str
    query: str = "" # noi dung/tham so chinh cua buoc, dung de phat hien trung lap

@dataclass
class Trajectory:
    name: str
    steps: list[Step]
    task_success: int # 0 hoac 1 - lay tu viec so khop final answer voi ground truth
    answer_quality: float # 0 -1 - gia lap diem LLM-judge (trong thuc te se goi judge that)
    tool_accuracy: float # 0 - 1 - ty le tool call dung ky thuat (dung tool, dung tham so, khong loi)

TOOL_COST_TABLE = {
    "search": (1.8, 0.0040),
    "calculator": (0.3, 0.0005),
    "kb_search": (1.2, 0.0030),
    "check_order_status": (0.9, 0.0020),
    "answer": (0.8, 0.0020),  # buoc sinh cau tra loi cuoi cung
}
    
SLA_LATENCY_SECONDS = 10.0
BUDGET_USD = 0.03

def detect_unnecessary_steps(steps: list[Step]) -> tuple[int, list[int]]:
    """Mot buoc la thua neu (tool, query) cua no da xuat hien o mot buoc TRUOC DO.
    Buoc 'answer' khong tinh vao day (luon can thiet)."""
    seen = set()
    unnecessary_idx = []
    for i,s in enumerate(steps):
        if s.tool == "answer":
            continue
        key = (s.tool, s.query)
        if key in seen:
            unnecessary_idx.append(i)
        else:
            seen.add(key)
    return len(unnecessary_idx), unnecessary_idx

def compute_latency_seconds(steps: list[Step]) -> float:
    return sum(TOOL_COST_TABLE[s.tool][0] for s in steps)

def compute_cost_usd(steps: list[Step]) -> float:
    return sum(TOOL_COST_TABLE[s.tool][1] for s in steps)

def build_metrics(traj: Trajectory) -> dict:
    n_unnecessary, idx = detect_unnecessary_steps(traj.steps)
    latency_s = compute_latency_seconds(traj.steps)
    cost = compute_cost_usd(traj.steps)
    return {
        "task_success": traj.task_success,
        "answer_quality": traj.answer_quality,
        "tool_accuracy": traj.tool_accuracy,
        "unnecessary_ratio": n_unnecessary / len(traj.steps),
        "unnecessary_idx": idx,
        "latency_seconds": round(latency_s, 2),
        "latency_norm": min(1.0, latency_s / SLA_LATENCY_SECONDS),
        "cost_usd": round(cost, 5),
        "cost_norm": min(1.0, cost / BUDGET_USD),
    }


## 2. Trọng số và hàm scoring (additive và gated)
score_additive dùng đúng công thức đề bài. score_gated là bản cải tiến: task_success đóng vai trò hệ số nhân (gate) thay vì cộng tuyến tính, để một trajectory trả lời SAI không thể được điểm bởi answer_quality/tool_accuracy cao.

In [2]:
@dataclass
class Weights:
    w_success: float = 0.35
    w_quality: float = 0.25
    w_tool_acc: float = 0.15
    w_unnecessary: float = 0.10
    w_latency: float = 0.10
    w_cost: float = 0.05

W = Weights()

def score_additive(m: dict, W: Weights = W) -> float:
    return (
        W.w_success * m["task_success"]
        + W.w_quality * m["answer_quality"]
        + W.w_tool_acc * m["tool_accuracy"] 
        - W.w_unnecessary * m["unnecessary_ratio"]
        - W.w_latency * m["latency_norm"]
        - W.w_cost * m["cost_norm"]
    )

def score_gated(m: dict, W: Weights = W) -> float:
    process_score = (
        W.w_success * m["task_success"]
        + W.w_quality * m["answer_quality"]
        + W.w_tool_acc * m["tool_accuracy"]
        - W.w_unnecessary * m["unnecessary_ratio"]
        - W.w_latency * m["latency_norm"]
        - W.w_cost * m["cost_norm"]
    )
    return m["task_success"] * process_score

def naive_step_count_score(steps: list[Step]) -> int:
    return -len(steps)

## 3. Case 1 - Agent A vs Agent B (đúng đề bài)
Agent A có 3 lần search trùng câu hỏi (thực sự lặp lại, không mang thêm thông tin) xen giữa Calculator, cộng thêm 1 search sau đó. Agent B chỉ search đúng 1 lần.

In [3]:
agent_a = Trajectory(
    name="Agent A",
    steps=[
        Step("search", "gia BTC hien tai"),
        Step("search", "gia BTC hien tai"),
        Step("search", "gia BTC hien tai"),
        Step("calculator", "0.5 x price"),
        Step("search", "gia BTC hien tai"),
        Step("answer", ""),
    ],
    task_success=1, answer_quality=0.9, tool_accuracy=1.0,
)

agent_b = Trajectory(
    name="Agent B",
    steps=[
        Step("search", "gia BTC hien tai"),
        Step("calculator", "0.5 x price"),
        Step("answer", ""),
    ],
    task_success=1, answer_quality=0.9, tool_accuracy=1.0
)

for traj in (agent_a, agent_b):
    m = build_metrics(traj)
    print(f"=== {traj.name} ===")
    print(f"  so buoc              : {len(traj.steps)}")
    print(f"  unnecessary_ratio    : {m['unnecessary_ratio']:.3f}  (idx thua: {m['unnecessary_idx']})")
    print(f"  latency              : {m['latency_seconds']}s -> norm {m['latency_norm']:.3f}")
    print(f"  cost                 : ${m['cost_usd']} -> norm {m['cost_norm']:.3f}")
    print(f"  Score (additive)     : {score_additive(m):.4f}")
    print(f"  Score (gated)        : {score_gated(m):.4f}")
    print(f"  Naive (-so buoc)     : {naive_step_count_score(traj.steps)}")
    print()

=== Agent A ===
  so buoc              : 6
  unnecessary_ratio    : 0.500  (idx thua: [1, 2, 4])
  latency              : 8.3s -> norm 0.830
  cost                 : $0.0185 -> norm 0.617
  Score (additive)     : 0.5612
  Score (gated)        : 0.5612
  Naive (-so buoc)     : -6

=== Agent B ===
  so buoc              : 3
  unnecessary_ratio    : 0.000  (idx thua: [])
  latency              : 2.9s -> norm 0.290
  cost                 : $0.0065 -> norm 0.217
  Score (additive)     : 0.6852
  Score (gated)        : 0.6852
  Naive (-so buoc)     : -3



Nhận xét: Score (additive lẫn gated) đều chọn Agent B cao hơn - nhưng lý do đúng nằm ở unnecessary_ratio và latency/cost thật sự thấp hơn, không phải vì B " có ít bước hơn".

## 4. Case 2 - Ví dụ Agent nhiều bước hơn nhưng TỐ T HƠN
Task: Kiểm tra bảo hành sản phẩm Y và tư vấn phương pháp nếu hết hạn

Agent C làm đủ 4 bước (không bước nào lặp) để đưa ra tư vấn đầy đủ.

Agent D chỉ trả lời chung chung, bỏ sót đúng phần user cần nhất.



In [4]:
agent_c = Trajectory(
    name="Agent C (nhieu buoc)",
    steps=[
        Step("kb_search", "warranty_policy"),
        Step("check_order_status", "ngay mua san pham Y"),
        Step("calculator", "so ngay con lai bao hanh"),
        Step("kb_search", "goi gia han bao hanh"),   # topic khac -> khong trung lap
        Step("answer", ""),
    ],
    task_success=1, answer_quality=0.95, tool_accuracy=1.0,
)

agent_d = Trajectory(
    name="Agent D (it buoc)",
    steps=[
        Step("kb_search", "warranty_policy"),
        Step("answer", ""),
    ],
    task_success=1, answer_quality=0.40, tool_accuracy=1.0,
)

for traj in (agent_c, agent_d):
    m = build_metrics(traj)
    print(f"=== {traj.name} ===")
    print(f"  so buoc              : {len(traj.steps)}")
    print(f"  unnecessary_ratio    : {m['unnecessary_ratio']:.3f}")
    print(f"  latency              : {m['latency_seconds']}s -> norm {m['latency_norm']:.3f}")
    print(f"  cost                 : ${m['cost_usd']} -> norm {m['cost_norm']:.3f}")
    print(f"  answer_quality       : {traj.answer_quality}")
    print(f"  Score (additive)     : {score_additive(m):.4f}")
    print(f"  Score (gated)        : {score_gated(m):.4f}")
    print(f"  Naive (-so buoc)     : {naive_step_count_score(traj.steps)}")
    print()

m_c, m_d = build_metrics(agent_c), build_metrics(agent_d)
principled_winner = "C" if score_additive(m_c) > score_additive(m_d) else "D"
naive_winner = "C" if naive_step_count_score(agent_c.steps) > naive_step_count_score(agent_d.steps) else "D"

print(f"Ket luan theo scoring co nguyen tac : Agent {principled_winner} thang")
print(f"Ket luan theo naive step-count      : Agent {naive_winner} thang")
assert principled_winner == "C" and naive_winner == "D", "Vi du phai the hien nghich ly nay!"
print("\n=> Xac nhan: naive heuristic (cang it buoc cang tot) cho ket luan SAI o case nay,")
print("   trong khi scoring co nguyen tac chon dung Agent C du no nhieu buoc hon.")

=== Agent C (nhieu buoc) ===
  so buoc              : 5
  unnecessary_ratio    : 0.000
  latency              : 4.4s -> norm 0.440
  cost                 : $0.0105 -> norm 0.350
  answer_quality       : 0.95
  Score (additive)     : 0.6760
  Score (gated)        : 0.6760
  Naive (-so buoc)     : -5

=== Agent D (it buoc) ===
  so buoc              : 2
  unnecessary_ratio    : 0.000
  latency              : 2.0s -> norm 0.200
  cost                 : $0.005 -> norm 0.167
  answer_quality       : 0.4
  Score (additive)     : 0.5717
  Score (gated)        : 0.5717
  Naive (-so buoc)     : -2

Ket luan theo scoring co nguyen tac : Agent C thang
Ket luan theo naive step-count      : Agent D thang

=> Xac nhan: naive heuristic (cang it buoc cang tot) cho ket luan SAI o case nay,
   trong khi scoring co nguyen tac chon dung Agent C du no nhieu buoc hon.


## 5. Vì sao cần task_success là gate - ví dụ Agent trả lời sai nhưng "quy trình đẹp"

Additive score có lỗ hổng: nếu answer_quality/tool_accuracy đủ cao, một agent trả lời sai vẫn có thể ra điểm dương. Gated score triệt tiêu hoàn toàn trường hợp này.

In [5]:
agent_e_wrong = Trajectory(
    name="Agent E (qui trinh dep nhung tra loi SAI)",
    steps=[
        Step("search", "gia ETH hien tai"),
        Step("calculator", "0.5 x price"),
        Step("answer", ""),
    ],
    task_success=0,          # cau tra loi cuoi cung SAI
    answer_quality=0.85,     # van trinh bay ro rang, co ve dang tin
    tool_accuracy=1.0,       # khong loi ky thuat nao trong qua trinh goi tool
)

m_e = build_metrics(agent_e_wrong)
print(f"Score additive (agent tra loi sai): {score_additive(m_e):.4f}")
print(f"Score gated    (agent tra loi sai): {score_gated(m_e):.4f}")

assert score_additive(m_e) > 0, "Additive score van duong du agent tra loi sai (day la lo hong!)"
assert score_gated(m_e) == 0.0, "Gated score phai bang 0 khi task_success = 0"
print("\n=> Additive score van > 0 (lo hong that su ton tai). Gated score dung bang 0,")
print("   dam bao khong co trajectory nao 'qua trinh dep' ma tra loi sai lai duoc diem duong.")

Score additive (agent tra loi sai): 0.3227
Score gated    (agent tra loi sai): 0.0000

=> Additive score van > 0 (lo hong that su ton tai). Gated score dung bang 0,
   dam bao khong co trajectory nao 'qua trinh dep' ma tra loi sai lai duoc diem duong.


## 6. Ghi chú / giới hạn
- answer_quality và tool_accuracy ở đây vẫn là input đã chấm sẵn (dại diện cho LLM-judge hoặc rule-based checker thật trong production) - bản thân việc chấm điểm chât lượng câu trả lời nằm ngoài phạm vi bài này
- detect_unnecessary_steps so khớp (tool, query) chính xác từng ký tự - production nên dùng similarity (embedding) để bắt được cả các query gần giống nhau (không chỉ trùng y hệt).
- bảng TOOL_COST_TABLE là hắng số giả định; trong production latency/cost phải đo thật từ log hệ thống (giống cách đã làm ở _estimate_cost trong bài 10) thay vì bảng trả cố định.

## 5 Câu hỏi bắt buộc

1. Failure mode lớn nhất: dùng heuristic "ít bước = tốt hơn" thay vì đo unnecessary_steps thật - cả 2 đều là proxy sai, dễ khen nhầm agent lười (bỏ qua verification cần thiết) hoặc agent lãng phí (lặp lại vô ích).
2. Biết hệ thống tốn lên khi nào: tỉ lệ unnecessary_steps giảm dần trong khi answer_quality không đổi hoặc tăng - tức agent học được cách đạt cùng chất lượng với ít lãng phí hơn, không phải cắt bước bừa bãi làm giảm chất lượng.
3. Metric nào cần theo dõi: theo dõi riêng answer_quality và unnecessary_steps thay vì chỉ nhìn score tổng - vì score tổng có thể ổn định trong khi hai xu hướng ngược nhau (quality giảm, waste cũng giảm) triệt tiêu lẫn nhau, che mất tín hiệu thật.
4. Có thực sự cần Agent không: Nếu task luôn cần đúng N bước cố định để verify đủ (như ví dụ Agent C: check policy -> check ngày mua -> tính toán -> tìm phương án), một workflow tất định N bước là đủ, không cần agent tự quyết định số bước; Agent thục sự cần khi số bước cần thiết thay đổi theo tình huống (có lúc cần verify thêm, có lúc không)
5. Kiến trúc đơn giản hơn: tách unnecessary_steps thành một detector riêng chạy sau khi có trajectory thay vì để agent tự "tự đánh giá" mình có thừa bước nào không - agent tự đánh giá dễ bias, một validator đọc lập đáng tin hơn.